In [ ]:
import json
import csv
import math
import time
import warnings
from pathlib import Path
from langchain_chroma import Chroma
from sentence_transformers import CrossEncoder

DOCS_DIR        = "../../data/parsed_json/"
CHROMA_DIR  = "../../data/vector_db_v4"
GOLDEN_SET_PATH = "../../eval/golden_dataset/golden_dataset_v2.csv"
COLLECTION_NAME = "rfp_docs"
EMBEDDING_MODEL = "bge-m3"
BATCH_SIZE      = 500
TOP_K           = 10
TOP_K_RETRIEVE  = 20

In [ ]:
GS_TO_DOCID = {
    "GKL_그룹웨어":            "D093",
    "KUSF_체육":               "D011",
    "강릉어선안전":             "D024",
    "경기_사회서비스":          "D087",
    "고려대학교_차세대포털":    "D008",
    "광주과기원_RCMS":          "D073",
    "광주과학기술원_학사시스템": "D039",
    "구미_육상":               "D018",
    "국립중앙의료원_응급":      "D069",
    "국민연금공단_이러닝":      "D049",
    "국민연금_멀티턴1":         "D050",
    "국민연금_멀티턴2":         "D050",
    "국민연금_멀티턴3":         "D050",
    "국방_대용량":             "D010",
    "기초과학연구원_극저온":    "D051",
    "나노종합_팹":             "D099",
    "대검찰청_홈페이지":        "D053",
    "민속박물관_아카이브":      "D090",
    "벤처협회_시스템":          "D086",
    "보험개발원_실손":          "D083",
    "봉화군_재난":             "D005",
    "부산관광_ERP":            "D037",
    "서민금융_채팅":            "D056",
    "서영대_교육":             "D045",
    "서울_디지털성범죄":        "D068",
    "서울_지도플랫폼":          "D040",
    "서울교육청_ISP":           "D084",
    "세종_인사":               "D088",
    "우즈벡_관개":             "D072",
    "울산_버스":               "D034",
    "인천_도시계획":            "D004",
    "인천_일자리":             "D030",
    "인천공항_ERP":            "D079",
    "적십자_재해복구":          "D095",
    "철도_ISP":               "D070",
    "통합정보시스템_충돌":      "D016",
    "평택_버스":               "D060",
    "해양박물관_자료":          "D066",
    "고려대_vs_광주과기원":     ["D008", "D039"],
    "버스_다중비교":            ["D034", "D060"],
    "재난안전_종합":            ["D005", "D007"],
    "철도_vs_인천_ISP":        ["D070", "D030"],
    "TEST": None, "unknown": None, "ISP_다중비교": None,
    "교육관련_다중문서": None, "문화_다중비교": None,
    "의료_다중문서": None, "재공고_종합": None,
    "신규_vs_고도화": None, "예산_최소_최대": None,
    "멀티턴_심화1": None, "멀티턴_심화2": None,
    "모른다_테스트1": None, "모른다_테스트2": None,
    "모른다_테스트3": None, "모른다_테스트4": None,
    "모른다_테스트5": None, "모른다_테스트6": None,
    "존재하지않는사업": None, "입찰마감_확인": None,
}


In [ ]:
def load_embedding_model(name: str):
    if name == "bge-m3":
        from langchain_huggingface import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(
            model_name="BAAI/bge-m3",
            model_kwargs={"device": "cuda"},
            encode_kwargs={"normalize_embeddings": True},
        )
    elif name == "text-embedding-3-small":
        from langchain_openai import OpenAIEmbeddings
        return OpenAIEmbeddings(model="text-embedding-3-small")
    else:
        raise ValueError(f"알 수 없는 임베딩 모델: {name}")

In [ ]:
def hit(retrieved_ids, target_ids, k):
    return any(tid in retrieved_ids[:k] for tid in target_ids)

In [ ]:
# 벡터 DB 로드
embedding_model = load_embedding_model(EMBEDDING_MODEL)
vs = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
    persist_directory=CHROMA_DIR,
)
print(f"벡터DB 로드 완료: {vs._collection.count()}개 청크")

In [ ]:
# Reranker 로드
reranker_minilm  = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cuda")
reranker_bge     = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cuda")
reranker_bge_base  = CrossEncoder("BAAI/bge-reranker-base", device="cuda")
reranker_bge_large = CrossEncoder("BAAI/bge-reranker-large", device="cuda")
print("Reranker 로드 완료")

In [ ]:
# 골든셋 로드
golden_set = []
with open(GOLDEN_SET_PATH, encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        golden_set.append(row)

golden_set = golden_set[:101]
print(f"골든셋 문항 수: {len(golden_set)}")

In [ ]:
# Reranker 함수
def rerank(query, hits, reranker_model, top_k):
    # hits: (doc, score) 리스트 -> reranker로 재정렬 후 doc_id 리스트 반환
    pairs = [(query, doc.page_content) for doc, _ in hits]
    scores = reranker_model.predict(pairs)
    ranked = sorted(zip(hits, scores), key=lambda x: x[1], reverse=True)
    return [doc.metadata.get("doc_id", "") for (doc, _), _ in ranked[:top_k]]

In [ ]:
STRATEGIES = [
    {"name": "baseline",            "reranker": None,           "top_k_retrieve": TOP_K},
    {"name": "reranker_minilm_k20",     "reranker": reranker_minilm,"top_k_retrieve": TOP_K_RETRIEVE},
    {"name": "reranker_bge_k20",        "reranker": reranker_bge,   "top_k_retrieve": TOP_K_RETRIEVE},
    {"name": "reranker_bge_base_k20",  "reranker": reranker_bge_base,  "top_k_retrieve": TOP_K_RETRIEVE},
    {"name": "reranker_bge_large_k20", "reranker": reranker_bge_large, "top_k_retrieve": TOP_K_RETRIEVE},
    {"name": "reranker_minilm_k10", "reranker": reranker_minilm,"top_k_retrieve": TOP_K},
    {"name": "reranker_bge_k10",    "reranker": reranker_bge,   "top_k_retrieve": TOP_K},
    {"name": "reranker_bge_base_k10",  "reranker": reranker_bge_base,  "top_k_retrieve": TOP_K},
    {"name": "reranker_bge_large_k10", "reranker": reranker_bge_large, "top_k_retrieve": TOP_K},
]

In [ ]:
# 전략별 실험 
eval_results = []

for strategy in STRATEGIES:
    print(f"\n{'='*50}")
    print(f"[전략] {strategy['name']}")

    recall3, recall5, recall10, mrr_scores, query_times = [], [], [], [], []
    skipped = 0

    for row in golden_set:
        gs_key = str(row["doc_id"])
        target = GS_TO_DOCID.get(gs_key)

        if target is None:
            skipped += 1
            continue

        target_ids = target if isinstance(target, list) else [target]

        t0 = time.time()

        if strategy["reranker"] is None:
            hits = vs.similarity_search_with_relevance_scores(row["question"], k=strategy["top_k_retrieve"])
            retrieved_doc_ids = [doc.metadata.get("doc_id", "") for doc, _ in hits]
        else:
            hits = vs.similarity_search_with_relevance_scores(row["question"], k=strategy["top_k_retrieve"])
            retrieved_doc_ids = rerank(row["question"], hits, strategy["reranker"], TOP_K)

        query_times.append(time.time() - t0)

        recall3.append(1.0 if hit(retrieved_doc_ids, target_ids, 3) else 0.0)
        recall5.append(1.0 if hit(retrieved_doc_ids, target_ids, 5) else 0.0)
        recall10.append(1.0 if hit(retrieved_doc_ids, target_ids, 10) else 0.0)

        rank = next(
            (i + 1 for i, d in enumerate(retrieved_doc_ids) if d in target_ids),
            None,
        )
        mrr_scores.append(1.0 / rank if rank else 0.0)

    n = len(recall3)
    print(f"  평가: {n}개 | 제외: {skipped}개")

    eval_results.append({
        "전략":      strategy["name"],
        "Recall@3":  round(sum(recall3)    / n, 4),
        "Recall@5":  round(sum(recall5)    / n, 4),
        "Recall@10": round(sum(recall10)   / n, 4),
        "MRR":       round(sum(mrr_scores) / n, 4),
        "avg_ms":    round(sum(query_times) / n * 1000, 1),
    })

In [ ]:
# 결과 출력
from IPython.display import Markdown, display

header = "| 전략 | Recall@3 | Recall@5 | Recall@10 | MRR | avg_ms |\n|------|----------|----------|-----------|-----|--------|"
rows = "\n".join(
    f"| {r['전략']} | {r['Recall@3']} | {r['Recall@5']} | {r['Recall@10']} | {r['MRR']} | {r['avg_ms']} |"
    for r in eval_results
)
display(Markdown(f"## Reranker 실험 결과\n\n{header}\n{rows}"))

reranker_bge_large_k20 선정

1. reranker_bge_large_k20이 최적
- Recall@10 0.3571 → 0.3810으로 약 7% 향상
- MRR 0.193 → 0.2366으로 약 23% 향상
- Recall@3 0.2262 → 0.2976으로 약 32% 향상
- avg_ms 899ms로 속도보다 정확도가 중요한 B2G 입찰 도메인 특성상 허용 가능한 수준으로 판단

2. 한국어 특화 모델(bge) vs 영어 특화 모델(minilm)
- minilm은 영어 특화라 한국어 공고문 재정렬 효과 미미
- bge 계열은 한국어를 이해해 정답을 상위로 올리는 데 효과적

### 지표 비교
| 전략 | Recall@3 | Recall@5 | Recall@10 | MRR | avg_ms |
|------|----------|----------|-----------|-----|--------|
| baseline | 0.2262 | 0.2857 | 0.3571 | 0.1930 | 24.7 |
| reranker_minilm_k20 | 0.2381 | 0.2857 | 0.3452 | 0.2141 | 96.1 |
| reranker_bge_k20 | 0.2024 | 0.3214 | 0.3929 | 0.2206 | 1921.1 |
| reranker_bge_base_k20 | 0.2738 | 0.3214 | 0.3690 | 0.2472 | 297.9 |
| reranker_bge_large_k20 | 0.2976 | 0.3095 | 0.3810 | 0.2366 | 899.2 |
| reranker_minilm_k10 | 0.2381 | 0.2619 | 0.3571 | 0.2214 | 61.0 |
| reranker_bge_k10 | 0.2262 | 0.3095 | 0.3571 | 0.1965 | 746.0 |
| reranker_bge_base_k10 | 0.2143 | 0.3095 | 0.3571 | 0.2121 | 153.1 |
| reranker_bge_large_k10 | 0.2500 | 0.2619 | 0.3571 | 0.2290 | 463.6 |